<a href="https://drawdown.org"><img style="float: left;" src="data/images/ProjectDrawdown.png"></a>
<br clear=left>&nbsp;<br/>

**[The World’s Leading Resource for Climate Solutions](https://drawdown.org)**

Our mission is to help the world reach “Drawdown”—the point in the future when levels of greenhouse gases in the atmosphere stop climbing and start to steadily decline, thereby stopping catastrophic climate change—as quickly, safely, and equitably as possible

+ There is a **<a href="https://youtu.be/RAQSnhm_tS4" target="_blank">video demonstrating</a>** how to use the system.  
+ If you run into trouble or need assistance, the **<a href="https://solutions.geekhold.com/services/discourse/" target="_blank">Community Forum</a>** can help.  

In [1]:
import ui.charts
(j, tabs) = ui.charts.get_ui()
tabs

In [134]:
import pandas as pd
import numpy as numpy
import os
import importlib
import ui.explorer
from pathlib import Path
import xlrd
import openpyxl
import pytest

In [135]:
import model.solution
import tools.excel_tools
import tests.excel_ranges

In [11]:
importlib.reload(model.solution)

<module 'model.solution' from 'C:\\Projects\\Project Drawdown\\solutions\\model\\solution.py'>

In [2]:
allsolns = model.solution.Solution.solution_list()

In [3]:
print(allsolns)

['afforestation', 'airplanes', 'altcement', 'bamboo', 'bikeinfrastructure', 'biochar', 'biogas', 'biogas_small', 'biomass', 'bioplastic', 'bottomtrawling', 'buildingautomation', 'carpooling', 'composting', 'concentratedsolar', 'conservationagriculture', 'coolroofs', 'districtheating', 'electricbikes', 'electricvehicles', 'farmlandrestoration', 'forestprotection', 'geothermal', 'grasslandprotection', 'greenroofs', 'heatpumps', 'highspeedrail', 'hybridcars', 'improvedcookstoves', 'improvedrice', 'indigenouspeoplesland', 'instreamhydro', 'insulation', 'irrigationefficiency', 'landfillmethane', 'leds_commercial', 'leds_residential', 'managedgrazing', 'mangroverestoration', 'masstransit', 'microwind', 'multistrataagroforestry', 'nuclear', 'nutrientmanagement', 'offshorewind', 'onshorewind', 'peatlands', 'perennialbioenergy', 'recycledpaper', 'refrigerants', 'regenerativeagriculture', 'riceintensification', 'ships', 'silvopasture', 'smartglass', 'smartthermostats', 'solarhotwater', 'solarpvr

In [4]:
broken = ['altcement','electricvehicles','hybridcars','trains','trucks']

In [3]:
# try simply to load a single scenario for each solution

for name in allsolns:
    try:
        print(name, end='')
        solution_module = importlib.import_module("solution." + name)
        print('..loaded module..', end='')
        obj = solution_module.Scenario()
        print('..loaded scenario..')
    except:
        print('..FAILED')


afforestation..loaded module....loaded scenario..
airplanes..loaded module....loaded scenario..
altcement..loaded module....loaded scenario..
bamboo..loaded module....loaded scenario..
bikeinfrastructure..loaded module....loaded scenario..
biochar..loaded module....loaded scenario..
biogas..loaded module....loaded scenario..
biogas_small..loaded module....loaded scenario..
biomass..loaded module....loaded scenario..
bioplastic..loaded module....loaded scenario..
bottomtrawling..loaded module....loaded scenario..
buildingautomation..loaded module....loaded scenario..
carpooling..loaded module....loaded scenario..
composting..loaded module....loaded scenario..
concentratedsolar..loaded module....loaded scenario..
conservationagriculture..loaded module....loaded scenario..
coolroofs..loaded module....loaded scenario..
districtheating..loaded module....loaded scenario..
electricbikes..loaded module....loaded scenario..
electricvehicles..loaded module....loaded scenario..
farmlandrestoratio

In [2]:
s = model.solution.Solution('trucks')

C:\ProgramData\Miniconda3\envs\dev_try2\lib\site-packages\pandas\core\arraylike.py:358: RuntimeWarning: invalid value encountered in minimum
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\ProgramData\Miniconda3\envs\dev_try2\lib\site-packages\numpy\core\fromnumeric.py:87: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
C:\ProgramData\Miniconda3\envs\dev_try2\lib\site-packages\pandas\core\arraylike.py:358: RuntimeWarning: invalid value encountered in minimum
  result = getattr(ufunc, method)(*inputs, **kwargs)
C:\ProgramData\Miniconda3\envs\dev_try2\lib\site-packages\numpy\core\fromnumeric.py:87: RuntimeWarning: invalid value encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


In [3]:
exp = ui.explorer.SolutionExplorer(s)

In [4]:
exp.show_co2_reduction()

In [2]:
?openpyxl.load_workbook

In [7]:
import openpyxl as oxl
from pathlib import Path

In [4]:
def parse_excel_range(range):
    """Return a tuple (first col, ncols, first row, nrows) from an excel range specified like "A2:B22" 
    Values returned are zero-indexed. """
    return (5,3,10,20)

In [68]:
def get_from_excel(path, sheet, excel_range, column_names = None):
    """Return the requested data from the spreadsheet as a pandas Dataframe.
    This is convenient shorthand for a common case; use `pd.read_excel` and `openpyxl` directly for more varied situations.
    
    `path`: a string path to open, or a file-like object to read from
    `sheet` the name or index of the sheet (tab) to read; if empty or None, the first sheet is assumed.
    `excel_range` the range to read, as a string in standard xls form (e.g "A2:B24")
    `column_names`: a list of column names to use for the result.  If empty or None, the first row of the range
    is assumed to contain the column names (and that row is not included in the data rows)

    Note: if the excel range contains an index column, you can assign that after the fact like this:
    `mydat = get_from_excel(...)
    mydat.index = mydat['index column name']`
    """
    (firstcol, firstrow, lastcol, lastrow) = oxl.utils.cell.range_boundaries(excel_range)
    nrows=lastrow-firstrow+1
    ncols=lastcol-firstcol+1
    
    if column_names:
        if len(column_names) != ncols:
            raise ValueError(f"Number of column names ({len(column_names)}) does not match number of columns to read ({ncols})")
        header = None
        skiprows = firstrow-1
        nrows = lastrow-firstrow+1
    else:
        header = firstrow-1
        skiprows = 0
        nrows = lastrow-firstrow
    
    wb = None
    try:
        wb = oxl.load_workbook(path, read_only=True, data_only=True, keep_links=False)
        return pd.read_excel(wb, sheet, 
            header=header, names=column_names, usecols=list(range(firstcol-1,lastcol)), 
            skiprows=skiprows, nrows=nrows, engine="openpyxl" )
    
    finally:
        if wb:
            wb.close()

In [136]:
f = Path('c:\\Projects\\sample.xlsm')

In [137]:
foo = get_from_excel(f, "Landfill Methane", "A7:D24") #, column_names=['year','A','B','C'])

In [138]:
foo

,Unnamed: 0,MMT Waste to Landfill by Year,Total Possible TWh Generation based on MMT Waste,Ambitious Case
0,2014,1285.786487,255.926425,23.994000
1,2015,1217.176621,242.270131,28.783102
2,2016,1227.044447,244.234250,32.713527
3,2017,1207.587545,240.361495,36.674692
4,2018,1186.280110,236.120406,40.675128
5,2019,1163.188415,231.524172,44.722994
6,2020,1138.160164,226.542482,49.016767
7,2021,1112.093326,221.354067,52.993875
8,2022,1084.185521,215.799222,57.234051
9,2023,1054.720375,209.934400,61.555742


In [63]:
foo.columns

Index(['Unnamed: 0', 'MMT Waste to Landfill by Year',
       'Total Possible TWh Generation based on MMT Waste', 'Ambitious Case'],
      dtype='object')

In [142]:
tools.excel_tools.range_shift("A7:D24", row_shift=-3, column_shift=2)

'C4:F21'

In [65]:
foo = foo.rename({'Unnamed: 0': 'Year'}, axis=1)

In [66]:
foo.index = foo['Year']

In [67]:
foo

,Year,MMT Waste to Landfill by Year,Total Possible TWh Generation based on MMT Waste,Ambitious Case
Year,,,,
2014,2014,1285.786487,255.926425,23.994000
2015,2015,1217.176621,242.270131,28.783102
2016,2016,1227.044447,244.234250,32.713527
2017,2017,1207.587545,240.361495,36.674692
2018,2018,1186.280110,236.120406,40.675128
2019,2019,1163.188415,231.524172,44.722994
2020,2020,1138.160164,226.542482,49.016767
2021,2021,1112.093326,221.354067,52.993875
2022,2022,1084.185521,215.799222,57.234051


In [42]:
opc.range_boundaries("A7:D24")

(1, 7, 4, 24)

In [80]:
def rename_column(df:pd.DataFrame, i, new_name):
    """Rename the `i`'th column in `df` to `new_name`.

    Returns a _new_ DataFrame.  Reassign to keep the modified df::
    
        mydf = rename_column(mydf, 2, 'some value')
    
    """
    return df.rename( columns={df.columns[i]: new_name} )

In [81]:
rename_column(foo,1,1)

,Year,1,Total Possible TWh Generation based on MMT Waste,Ambitious Case
Year,,,,
2014,2014,1285.786487,255.926425,23.994000
2015,2015,1217.176621,242.270131,28.783102
2016,2016,1227.044447,244.234250,32.713527
2017,2017,1207.587545,240.361495,36.674692
2018,2018,1186.280110,236.120406,40.675128
2019,2019,1163.188415,231.524172,44.722994
2020,2020,1138.160164,226.542482,49.016767
2021,2021,1112.093326,221.354067,52.993875
2022,2022,1084.185521,215.799222,57.234051


In [82]:
?rename_column

In [83]:
def rename_all_columns(df:pd.DataFrame, name_list):
    """Rename all the columns of `df`.  If any name in the list is None or NaN, that column
    is omitted from the result.

    Returns a _new_ DataFrame.  Reassign to keep the modified df::
    
        mydf = rename_all_columns(mydf, ['year', None, 'A', 'B', 'C'])
    """
    if len(name_list) != len(df.columns):
        raise ValueError(f"Number of column names provided ({len(name_list)}) doesn't match number of columns in df ({len(df.columns)})")
    
    keepcols = [ x for x in name_list if not pd.isna(x) ]
    mapcols = dict( zip( df.columns, name_list ) )
    mappeddf = df.rename(columns=mapcols)
    return mappeddf[keepcols]

,NaN,MMT Waste to Landfill by Year,Total Possible TWh Generation based on MMT Waste,Ambitious Case
Year,,,,
2014,2014,1285.786487,255.926425,23.994000
2015,2015,1217.176621,242.270131,28.783102
2016,2016,1227.044447,244.234250,32.713527
2017,2017,1207.587545,240.361495,36.674692
2018,2018,1186.280110,236.120406,40.675128
2019,2019,1163.188415,231.524172,44.722994
2020,2020,1138.160164,226.542482,49.016767
2021,2021,1112.093326,221.354067,52.993875
2022,2022,1084.185521,215.799222,57.234051


In [89]:
bar = Path('.')

In [90]:
bar

WindowsPath('.')

In [91]:
bar.parents

<WindowsPath.parents>

In [92]:
bar.absolute()

WindowsPath('C:/Projects/Project Drawdown/solutions')

In [93]:
bar.absolute().parents

<WindowsPath.parents>

In [94]:
bar.absolute().parents()

TypeError: '_PathParents' object is not callable

In [95]:
bar.absolute().parents[0]

WindowsPath('C:/Projects/Project Drawdown')

In [100]:
import numpy as np

In [101]:
bar = pd.DataFrame(np.random.randn(6, 4), columns=list("ABCD"))

In [102]:
bar

,A,B,C,D
0,-0.355590,1.727817,-0.519733,-0.876550
1,-0.903391,-0.613079,-0.940780,1.178210
2,0.058866,0.121847,-2.045014,0.454803
3,0.181154,0.809661,-0.774325,0.306181
4,-1.450060,0.069794,1.539890,0.105334
5,0.871078,-0.527859,-0.145403,0.809522


In [104]:
baseline=pd.Series([-0.2, 0.0, 0.1, 0.2], index=["A","B","C","D"])

In [105]:
baseline

A   -0.2
B    0.0
C    0.1
D    0.2
dtype: float64

In [106]:
bar.mask(bar<baseline)

,A,B,C,D
0,NaN,1.727817,NaN,NaN
1,NaN,NaN,NaN,1.178210
2,0.058866,0.121847,NaN,0.454803
3,0.181154,0.809661,NaN,0.306181
4,NaN,0.069794,1.53989,NaN
5,0.871078,NaN,NaN,0.809522


In [107]:
bar.mask(bar<baseline,other=True)

,A,B,C,D
0,True,1.727817,True,True
1,True,True,True,1.17821
2,0.058866,0.121847,True,0.454803
3,0.181154,0.809661,True,0.306181
4,True,0.069794,1.53989,True
5,0.871078,True,True,0.809522


In [108]:
bar.mask(bar<baseline,other=True).where(bar<baseline,other=False)

,A,B,C,D
0,True,False,True,True
1,True,True,True,False
2,False,False,True,False
3,False,False,True,False
4,True,False,False,True
5,False,True,True,False


In [109]:
bar<baseline

,A,B,C,D
0,True,False,True,True
1,True,True,True,False
2,False,False,True,False
3,False,False,True,False
4,True,False,False,True
5,False,True,True,False


In [114]:
baseline=baseline.reindex(['D','C','B'])

In [115]:
baseline

D    0.2
C    0.1
B    0.0
dtype: float64

In [116]:
bar < baseline

<ipython-input-116-8456526739f9>:1: FutureWarning: Automatic reindexing on DataFrame vs Series comparisons is deprecated and will raise ValueError in a future version.  Do `left, right = left.align(right, axis=1, copy=False)` before e.g. `left == right`
  bar < baseline


,A,B,C,D
0,False,False,True,True
1,False,True,True,False
2,False,False,True,False
3,False,False,True,False
4,False,False,False,True
5,False,True,True,False


KeyError: 'key of type tuple not found and not a MultiIndex'

In [117]:
import pytest

In [118]:
pytest.approx(2.0)

2.0 ± 2.0e-06

In [119]:
2.000002 == pytest.approx(2.0)

True

In [120]:
pytest.approx(None)

None

In [121]:
pytest.approx(np.nan)

nan ± ???

In [122]:
np.isna(np.NaN)

AttributeError: module 'numpy' has no attribute 'isna'

In [123]:
pd.isna(np.NaN)

True

In [124]:
np.isInf(np.NaN)

AttributeError: module 'numpy' has no attribute 'isInf'

In [125]:
np.isinf(np.NaN)

False

In [126]:
np.isnan(20)

False

In [127]:
import numba

In [130]:
print(numba.NumbaWarning)

<class 'numba.core.errors.NumbaWarning'>


In [131]:
??numba.NumbaWarning

In [132]:
myd = {'a': "foo"}

In [133]:
len(myd)

1